# Clase 227 — GDPR y AI Act: mini-toolkit programático

Esta clase es legal/operativa. El notebook implementa un **compliance toolkit** que un equipo de datos puede correr antes de pasar un modelo a producción.

Solo `numpy`, `pandas`, `scikit-learn`, `re`. Seed 42. Sin datos reales — generamos un sintético de decisiones de crédito.

In [ ]:
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)
n = 1000

df = pd.DataFrame({
    'user_id':   np.arange(n),
    'email':     [f'user{i}@example.com' for i in range(n)],
    'dni':       [f'{rng.integers(10_000_000, 99_999_999)}{chr(rng.integers(65, 91))}' for _ in range(n)],
    'age':       rng.integers(18, 80, n),
    'salary':    rng.normal(35_000, 12_000, n).clip(8_000, 200_000).round(),
    'is_minority': rng.integers(0, 2, n),
    'past_default': rng.integers(0, 2, n),
})
df['approved'] = ((df['salary'] > 28_000) & (df['past_default'] == 0)).astype(int)
df['approved'] = np.where(rng.random(n) < 0.1, 1 - df['approved'], df['approved'])
df.head(3)

## 1. Clasificación de riesgo según AI Act (Anexo III)

Lookup table simplificada de los 4 niveles. En producción, esto lo cierra legal con el caso de uso real.

In [ ]:
AI_ACT_RISK = {
    'prohibido': [
        'social scoring', 'manipulacion cognitiva', 'scraping facial indiscriminado',
        'reconocimiento emociones trabajo', 'reconocimiento emociones educacion',
        'categorizacion biometrica datos sensibles',
    ],
    'alto': [
        'biometria', 'infraestructura critica', 'educacion admision',
        'cv screening', 'reclutamiento', 'credito', 'seguros vida salud',
        'servicios esenciales', 'aplicacion ley', 'migracion asilo', 'justicia',
    ],
    'limitado': ['chatbot', 'generacion contenido', 'deepfake'],
    'minimo':   ['filtro spam', 'recomendador musica', 'videojuego', 'optimizacion logistica'],
}

def _obligations_for(level):
    return {
        'prohibido': ['NO desplegar — prohibido por Art. 5 AI Act'],
        'alto': ['gestion de riesgos', 'gobernanza de datos', 'documentacion tecnica',
                 'logs/registros', 'transparencia al usuario', 'supervision humana (Art. 14)',
                 'robustez y ciberseguridad', 'marcado CE'],
        'limitado': ['informar al usuario que interactua con IA',
                     'marcar contenido generado/deepfake'],
        'minimo':   ['recomendado: codigos de conducta voluntarios'],
    }[level]

def is_high_risk_use_case(use_case: str) -> dict:
    s = use_case.lower()
    for level, keywords in AI_ACT_RISK.items():
        if any(k in s for k in keywords):
            return {'use_case': use_case, 'risk_level': level,
                    'obligations': _obligations_for(level)}
    return {'use_case': use_case, 'risk_level': 'no clasificado',
            'obligations': ['revisar manualmente con DPO/legal']}

for uc in ['CV screening para puesto senior', 'Credito al consumo', 'Chatbot soporte',
           'Recomendador musica', 'Social scoring ciudadano']:
    print(json.dumps(is_high_risk_use_case(uc), ensure_ascii=False))

## 2. DPIA checklist (Art. 35 GDPR)

Dado un proyecto, devuelve las obligaciones GDPR aplicables.

In [ ]:
def dpia_checklist(project: dict) -> dict:
    """Recibe descripcion del proyecto, retorna obligaciones GDPR.

    Keys esperadas: sensitive_categories, automated_decisions, scale,
                    vulnerable_subjects, systematic_monitoring, cross_border."""
    obligations, dpia_required = [], False
    if project.get('sensitive_categories'):
        obligations.append('Art. 9: base legal reforzada (consentimiento explicito o excepcion)')
        dpia_required = True
    if project.get('automated_decisions'):
        obligations.append('Art. 22: derecho a no decision solo automatizada + intervencion humana')
        dpia_required = True
    if project.get('scale') == 'large':
        obligations.append('Art. 37: designar DPO (Data Protection Officer)')
        dpia_required = True
    if project.get('vulnerable_subjects'):
        obligations.append('Art. 35: DPIA reforzada (menores, pacientes, empleados)')
        dpia_required = True
    if project.get('systematic_monitoring'):
        obligations.append('Art. 35.3(c): DPIA por monitoreo sistematico')
        dpia_required = True
    if project.get('cross_border'):
        obligations.append('Cap. V: clausulas tipo (SCC) para transferencias fuera UE')
    obligations.extend([
        'Art. 5: principios (minimizacion, exactitud, limitacion plazo, integridad)',
        'Art. 6: identificar base legal',
        'Art. 13-14: informacion al titular',
        'Art. 15-20: derechos (acceso, rectificacion, supresion, portabilidad)',
        'Art. 32: medidas tecnicas y organizativas',
    ])
    return {'dpia_required': dpia_required, 'obligations': obligations}

credit_project = {
    'sensitive_categories': False, 'automated_decisions': True,
    'scale': 'large', 'vulnerable_subjects': False,
    'systematic_monitoring': True, 'cross_border': False,
}
print(json.dumps(dpia_checklist(credit_project), ensure_ascii=False, indent=2))

## 3. Right to be forgotten (Art. 17 GDPR)

In [ ]:
def right_to_be_forgotten(df: pd.DataFrame, user_id: int):
    """Elimina al usuario del DataFrame. Devuelve (df_actualizado, audit_record)."""
    mask = df['user_id'] == user_id
    if not mask.any():
        return df, {'status': 'not_found', 'user_id': int(user_id)}
    audit = {
        'status': 'deleted',
        'user_id': int(user_id),
        'deleted_at': datetime.now(timezone.utc).isoformat(),
        'columns_affected': list(df.columns),
        'rows_removed': int(mask.sum()),
        'legal_basis': 'GDPR Art. 17 — derecho a la supresion',
        'follow_up': ['propagar a backups', 'propagar a logs', 're-entrenar modelo si aplica'],
    }
    return df.loc[~mask].reset_index(drop=True), audit

df2, audit = right_to_be_forgotten(df, user_id=42)
print(f'antes: {len(df)} filas, despues: {len(df2)} filas')
print(json.dumps(audit, ensure_ascii=False, indent=2))
assert (df2['user_id'] == 42).sum() == 0

## 4. Data minimization audit (Art. 5.1.c)

In [ ]:
PII_PATTERNS = {
    'email':  re.compile(r'^[\w\.\-]+@[\w\.\-]+\.\w+$'),
    'dni_es': re.compile(r'^\d{8}[A-Z]$'),
    'phone':  re.compile(r'^\+?\d{9,15}$'),
}

def data_minimization_audit(df: pd.DataFrame, sample: int = 50):
    findings = []
    for col in df.select_dtypes(include='object').columns:
        values = df[col].dropna().astype(str).head(sample)
        for label, pat in PII_PATTERNS.items():
            hits = sum(bool(pat.match(v)) for v in values)
            if hits / max(len(values), 1) > 0.5:
                findings.append({
                    'column': col, 'type': label,
                    'cardinality': int(df[col].nunique()),
                    'recommendation': 'hash con salt o remover si no es necesario para el modelo',
                })
    return findings

for f in data_minimization_audit(df):
    print(f)

## 5. Model card mínima (Mitchell et al., 2019)

Documentación obligatoria para alto riesgo (AI Act Art. 11 + Anexo IV).

In [ ]:
def model_card(model, X_train, y_train, sensitive_attr: pd.Series, name: str) -> dict:
    y_pred = model.predict(X_train)
    perf_per_group = {}
    for group_val in sorted(sensitive_attr.unique()):
        mask = (sensitive_attr == group_val).values
        perf_per_group[f'group_{group_val}'] = {
            'n': int(mask.sum()),
            'accuracy': round(accuracy_score(y_train[mask], y_pred[mask]), 4),
            'positive_rate': round(float(y_pred[mask].mean()), 4),
        }
    return {
        'model_name': name,
        'created_at': datetime.now(timezone.utc).isoformat(),
        'intended_use': 'Scoring de credito al consumo (alto riesgo AI Act Anexo III)',
        'out_of_scope': ['hipotecas', 'credito empresarial', 'menores'],
        'training_data': {'n_rows': int(len(X_train)),
                          'features': list(X_train.columns),
                          'period': '2024-Q1 sintetico'},
        'metrics_overall': {'accuracy': round(accuracy_score(y_train, y_pred), 4)},
        'metrics_per_group': perf_per_group,
        'limitations': ['datos sinteticos', 'no testeado en distribucion real',
                        'no incluye factores macroeconomicos'],
        'owner': 'data-team@example.com',
    }

features = ['age', 'salary', 'past_default']
X, y = df[features], df['approved']
model = LogisticRegression(max_iter=1000).fit(X, y)
card = model_card(model, X, y, df['is_minority'], name='credit_scorer_v1')
print(json.dumps(card, ensure_ascii=False, indent=2))

## 6. Human-in-the-loop (Art. 22 GDPR + Art. 14 AI Act)

In [ ]:
def human_in_the_loop_required(decision_score: float,
                                threshold_low: float = 0.4,
                                threshold_high: float = 0.7) -> dict:
    """Determina si una decision requiere revision humana."""
    if threshold_low <= decision_score <= threshold_high:
        return {'decision': 'pending', 'route': 'human_review',
                'reason': 'score en zona borderline — Art. 22 GDPR + Art. 14 AI Act'}
    decision = 'approve' if decision_score > threshold_high else 'reject'
    return {'decision': decision, 'route': 'automated',
            'reason': 'fuera de banda gris', 'appeal_available': True}

for s in [0.92, 0.55, 0.30, 0.71]:
    print(f'score={s} -> {human_in_the_loop_required(s)}')

## 7. Compliance report — pipeline completo

In [ ]:
def compliance_report(df, model, features, sensitive_col, use_case, project_meta):
    report = {}
    print('=' * 60); print(f'COMPLIANCE REPORT — {use_case}'); print('=' * 60)

    report['ai_act'] = is_high_risk_use_case(use_case)
    print(f"\n[1] AI Act risk: {report['ai_act']['risk_level'].upper()}")
    for o in report['ai_act']['obligations']:
        print(f'    - {o}')

    report['dpia'] = dpia_checklist(project_meta)
    print(f"\n[2] DPIA requerida: {report['dpia']['dpia_required']}")
    print(f"    Obligaciones GDPR: {len(report['dpia']['obligations'])} items")

    report['pii'] = data_minimization_audit(df)
    print(f"\n[3] PII detectada: {len(report['pii'])} columnas con dato personal directo")
    for f in report['pii']:
        print(f"    - {f['column']} ({f['type']})")

    report['model_card'] = model_card(model, df[features], df['approved'],
                                       df[sensitive_col], name='credit_scorer_v1')
    print(f"\n[4] Model card emitida — accuracy overall={report['model_card']['metrics_overall']['accuracy']}")
    for g, m in report['model_card']['metrics_per_group'].items():
        print(f'    {g}: acc={m["accuracy"]}, positive_rate={m["positive_rate"]}')

    scores = model.predict_proba(df[features].head(5))[:, 1]
    report['hitl_sample'] = [human_in_the_loop_required(float(s)) for s in scores]
    n_human = sum(d['route'] == 'human_review' for d in report['hitl_sample'])
    print(f'\n[5] Human-in-the-loop sample: {n_human}/5 a revision humana')

    _, audit = right_to_be_forgotten(df, user_id=int(df['user_id'].iloc[0]))
    report['rtbf_demo'] = audit
    print(f"\n[6] Right to be forgotten demo: user_{audit['user_id']} -> {audit['status']}")

    print('\n' + '=' * 60); print('FIN DEL REPORTE'); print('=' * 60)
    return report

_ = compliance_report(
    df=df, model=model, features=features,
    sensitive_col='is_minority',
    use_case='Credito al consumo automatizado',
    project_meta=credit_project,
)

## Ejercicio guiado

1. Agregá tu propio caso de uso a `AI_ACT_RISK` y verificá la clasificación.
2. Implementá `right_to_access(df, user_id)` (Art. 15) que devuelva **todo** lo que sabés del titular en formato portable (JSON).
3. Extendé `data_minimization_audit` con patrones de IBAN, tarjeta (Luhn), y direcciones IP.
4. Calculá **demographic parity gap** sobre `is_minority` y agregalo a la model card (link a Clase 226).
5. Convertí el reporte de la celda 7 en un Markdown auto-generado listo para adjuntar a un PR.

## Conclusiones

- GDPR regula el **dato**: base legal, derechos del titular, DPIA, multas hasta 20 M€ / 4%.
- AI Act regula el **sistema**: pirámide de riesgo, obligaciones por nivel, multas hasta 35 M€ / 7%.
- Compliance no es papel: se programa. Un toolkit de 6 funciones cubre el 80% del checklist diario.
- Art. 22 GDPR + Art. 14 AI Act: para alto impacto, **siempre** dejar puerta a intervención humana significativa.